# DEXTRA · 07 — Evening precursors: does the end of today warn of a severe tomorrow?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Aduitya9/DEXTRA/blob/main/notebooks/07_evening_precursors.ipynb)

Notebook 06 found where severe-day forecasting actually fails. In Delhi every forecaster does well. Everywhere else every forecaster — trained, zero-shot or fixed — catches few severe days, and **most of those days are onsets**: severe on the target day, not on the day the forecast was made (332 of 524 outside Delhi in 2024–25). Every model here reads each day as one daily average, and a daily average dilutes an evening surge with a clean morning.

The hourly CPCB data keeps the shape of the day. On the **training years alone** (2019–2023, outside Delhi), among days in the 200–300 band where onsets start, the last three hours of the evening separate tomorrow's onsets with AUC 0.773 against 0.689 for the day's AQI. That was found without looking at a single test-year day, so it can be tested honestly: give each model the evening, and see whether it then ranks severe days better on cities it has never seen.

### The evening features
Nine per city-day, built by `scripts/build_evening_features.py` from the hourly station files (which stay on the machine that downloaded them) and the hourly ERA5 fields: PM2.5 and PM10 over 18:00–24:00 and over its last three hours, the rise from the same day's morning, the evening's PM sub-index on the AQI scale, and ERA5's evening boundary-layer height and wind. Every value comes from day *t*'s own hours.

### What is compared — fixed before any result is read
Three model families, each **with and without** the evening features: notebook 06's all-token head (three seeds), gradient boosting, and Chronos-2 (zero-shot, the evening as past covariates). The "without" twins are notebook 06's own saved forecasts — same windows, same seeds, same recipe — so the only difference is the evening.

**Primary measure:** pooled leave-one-city-out average precision for AQI above 300, with minus without, for each family; a 95% interval and a p-value from resampling whole held-out cities; Holm-corrected across the three families. Average precision rather than notebook 06's alarm F1, because notebook 06 showed the alarm level alone can move one city's F1 from 0.41 to 0.09 — average precision asks only how well a forecast ranks the severe days. The evening features **help a family** if its corrected p is below 0.05 and the difference is positive. Onsets on their own, the five main-split unseen cities, the cities other than Delhi, RMSE and alarm F1 are reported beside it.

### Runtime
**GPU, about 25 minutes on an A100 or an hour on a T4**: three main-split runs, gradient boosting, one Chronos-2 pass (a 480 MB download), and fifteen folds of three runs each. Every piece is saved when it finishes.

In [ ]:
# Leave empty to read from the HF_DATASET_REPO Colab secret.
HF_DATASET_REPO = ""
GITHUB_REPO_URL = "https://github.com/Aduitya9/DEXTRA.git"

SEEDS = [42, 43, 44]              # notebook 06's, so every run has its twin without the evening

# Architecture and training: notebook 06's all-token head, unchanged.
D_MODEL, N_HEADS, N_LAYERS, DROPOUT = 128, 8, 2, 0.3
EPOCHS, BATCH_SIZE, LEARNING_RATE, WEIGHT_DECAY, PATIENCE = 30, 256, 3e-4, 1e-3, 10

# Chronos-2 exactly as notebook 06 ran it, now with the evening as past covariates.
FM_MODEL, FM_CONTEXT = "amazon/chronos-2", 512

LOCO_CITIES = None                # None holds out all fifteen, as notebook 06 did
BOOTSTRAP_DRAWS = 1000

# Safety
TIME_BUDGET_HOURS = 10.5
CHECKPOINT_EVERY_MINUTES = 4

SAVE_TO_HF = True

### If Colab disconnects
The same protection as notebooks 05 and 06: each piece — a main-split run, gradient boosting, the Chronos-2 pass, a whole leave-one-city-out fold — is saved in one commit when it finishes, and re-running skips everything finished. A main-split run cut off part-way resumes from its last checkpoint; a fold cut off part-way starts again (about a minute on an A100). If you are disconnected: reconnect and **Run all**. Near Colab's 12-hour limit: **Runtime → Disconnect and delete runtime**, then **Run all**. Keep the tab open and your computer awake while it runs.

## Secrets
`HF_TOKEN` and `HF_DATASET_REPO`, as in the earlier notebooks.

In [ ]:
from google.colab import userdata

def colab_secret(name, required=True):
    try:
        value = userdata.get(name)
    except Exception:
        value = None
    if required and not value:
        raise RuntimeError(
            f"Colab secret '{name}' is missing or this notebook can't read it. Open the key icon "
            "(Secrets) in the left sidebar, add it, and switch on 'Notebook access'."
        )
    return value

HF_TOKEN = colab_secret("HF_TOKEN")
HF_DATASET_REPO = HF_DATASET_REPO or colab_secret("HF_DATASET_REPO")
print("Dataset repo:", HF_DATASET_REPO)

In [ ]:
%pip install -q "huggingface_hub>=0.23.0" pyarrow "chronos-forecasting>=2.3,<3"

In [ ]:
import io
import os
import subprocess
import sys
import tempfile
import time
from pathlib import Path

REPO_DIR = "/content/DEXTRA"
if os.path.isdir(os.path.join(REPO_DIR, ".git")):
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only", "-q"], check=True)
else:
    subprocess.run(["git", "clone", "-q", GITHUB_REPO_URL, REPO_DIR], check=True)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

import numpy as np
import pandas as pd
import torch
from huggingface_hub.utils import disable_progress_bars, enable_progress_bars
from sklearn.metrics import roc_auc_score
from src import config
from src.data import assemble, evening
from src.evaluation import alarms, gap
from src.models import baselines, foundation, itransformer, met_moe, sequences, training, windows
from src.utils.hf_store import HFStore

disable_progress_bars()
device = itransformer.pick_device()
store = HFStore(HF_DATASET_REPO, token=HF_TOKEN)
needed = ["processed/evening_features.parquet", "processed/gap_main_forecasts.parquet",
          "processed/gap_loco_forecasts.parquet"]
missing = [p for p in needed if not store.exists(p)]
if missing:
    raise RuntimeError(f"Missing from the dataset repo: {missing}. The evening features come from "
                       "scripts/build_evening_features.py --upload, the forecast files from notebook 06.")

def by_day(frame):
    frame["date"] = pd.to_datetime(frame["date"]).astype("datetime64[ns]")
    return frame

EVENING = evening.FEATURES
WITH = list(config.ALL_FEATURES) + EVENING
daily = by_day(store.read_df("processed/city_daily_features.parquet"))
daily = daily.merge(by_day(store.read_df("processed/evening_features.parquet")), on=["city", "date"], how="left")

# Recomputed from the current config rather than trusted as saved.
saved = daily["split"].copy() if "split" in daily.columns else None
daily["split"] = assemble.assign_split(daily)
differing = int((saved != daily["split"]).sum()) if saved is not None else 0
if differing:
    print(f"! {differing:,} rows differ from the saved splits; re-run notebook 02.")

# The 12-hour ceiling counts from when this VM booted, not from this cell.
try:
    session_age = float(open("/proc/uptime").read().split()[0])
except (OSError, ValueError, IndexError):
    session_age = 0.0
if not 0.0 <= session_age < 12 * 3600:
    session_age = 0.0
deadline = time.monotonic() + max(0.0, TIME_BUDGET_HOURS * 3600 - session_age)

registry = training.RunRegistry(store, "runs/07")
cities = sorted(daily.city.unique()) if LOCO_CITIES is None else list(LOCO_CITIES)
fold_run = lambda city: "loco-" + city.replace(" ", "_")
planned = ([f"main-evening_seed{seed}" for seed in SEEDS] + ["main-gbm-evening", "fm-chronos-2-evening"]
           + [fold_run(city) for city in cities])
finished = [name for name in planned if registry.done(name)]

print(f"device: {device}" + (f" ({torch.cuda.get_device_name(0)})" if device.type == "cuda" else ""))
if device.type != "cuda":
    print("! No GPU: this will take hours. Runtime -> Change runtime type -> a GPU, then Run all.")
print(f"{len(daily):,} city-days, {daily.city.nunique()} cities, {len(EVENING)} evening features")
print(f"{len(finished)} of {len(planned)} pieces already finished"
      + (" -- resuming where the last session stopped" if finished else ""))
print(f"time budget left this session: {(deadline - time.monotonic()) / 3600:.1f} h")

## A · The evening features, and the signal on the training years
How much of the record each feature covers, and the evidence the test rests on — computed here from the **training years only** (2019–2023), outside Delhi, so that nothing below has seen a test-year day. AUC is shown as max(AUC, 1 − AUC): a low evening boundary layer warns as surely as a high evening PM2.5.

In [ ]:
display(daily[EVENING].notna().mean().rename("share of city-days present").round(3).to_frame())

past = daily[(daily.date <= pd.Timestamp(config.TRAIN_END_DATE)) & (daily.city != "Delhi")]
past = past.sort_values(["city", "date"])
past = past.assign(tomorrow=past.groupby("city").AQI.shift(-1)).dropna(subset=["AQI", "tomorrow"])
onset = (past.tomorrow > config.RARE_AQI_THRESHOLD) & (past.AQI <= config.RARE_AQI_THRESHOLD)
band = past.AQI.between(config.HAZARDOUS_AQI_THRESHOLD, config.RARE_AQI_THRESHOLD)
rows = []
for column in ["AQI"] + EVENING:
    values = past.loc[band, column]
    auc = roc_auc_score(onset[band], values.fillna(values.median()))
    rows.append({"feature": column, "AUC for tomorrow's onset": max(auc, 1 - auc)})
print(f"Training years, outside Delhi, days in the 200-300 band: {int(band.sum()):,} days, "
      f"{int(onset[band].sum())} onsets")
display(pd.DataFrame(rows).set_index("feature").round(3))
quartile = pd.qcut(past.loc[band, "pm25_late"], 4, labels=["lowest", "second", "third", "highest"])
print("Onset rate by the last three evening hours' PM2.5, today's AQI held in the 200-300 band:")
display(pd.DataFrame({"days": quartile.value_counts().sort_index(),
                      "onset rate": onset[band].groupby(quartile, observed=True).mean().round(3)}))

## B · The main split: each family with the evening
The all-token head, retrained from notebook 06's three seeds with the nine evening features as extra variates; gradient boosting refitted with them as extra columns; and Chronos-2 run once more with them as past covariates. Their twins without the evening are read from notebook 06 in section D.

In [ ]:
X, y, index = sequences.build_sequences(daily, features=WITH)
variates = sequences.feature_names(daily, WITH)
target_index = variates.index(sequences.TARGET)
gate_indices, gate_logged = met_moe.gate_layout(variates)
is_train, is_val = (index.split == "train").to_numpy(), (index.split == "val").to_numpy()
HORIZONS = [f"h{h + 1}" for h in range(y.shape[1])]
scored = (index.last_target_date > pd.Timestamp(config.TRAIN_END_DATE)).to_numpy()
origins = index.loc[scored, ["city", "date"]].reset_index(drop=True)
KEYS = ["city", "date", "horizon"]
print(f"X {X.shape}: {len(variates)} variates, {len(EVENING)} of them the evening's")


def save_run(name, result, frames=None, weights=None):
    """Record a finished piece with its forecasts (and weights) in one commit."""
    if not SAVE_TO_HF:
        return
    with tempfile.TemporaryDirectory() as tmp:
        attachments = {}
        for filename, frame in (frames or {}).items():
            attachments[filename] = Path(tmp) / filename
            frame.to_parquet(attachments[filename], index=False)
        if weights is not None:
            attachments[f"{name}.pt"] = Path(tmp) / f"{name}.pt"
            torch.save(weights, attachments[f"{name}.pt"])
        registry.record(name, result, attachments=attachments)


def load_frame(name):
    return by_day(pd.read_parquet(io.BytesIO(store.read_bytes(registry.attachment(f"{name}.parquet")))))


def build_head():
    return met_moe.MetMoE(X.shape[2], config.LOOKBACK_WINDOW, config.FORECAST_HORIZON, target_index,
                          gate_indices, gate_logged, router="none", d_model=D_MODEL, n_heads=N_HEADS,
                          n_layers=N_LAYERS, dropout=DROPOUT)


def fit(seed, X_train, y_train, val_loader, checkpoint=None, log=print):
    itransformer.set_seed(seed)
    checkpointer = (training.Checkpointer(store, checkpoint, every_seconds=CHECKPOINT_EVERY_MINUTES * 60)
                    if checkpoint else None)
    return training.train(build_head(), itransformer.make_loader(X_train, y_train, BATCH_SIZE, shuffle=True),
                          val_loader, device, epochs=EPOCHS, lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY,
                          patience=PATIENCE, checkpointer=checkpointer, deadline=deadline,
                          save_finished=False, log=log)


def best_epoch(history):
    return min(history, key=lambda h: h["val_rmse"])["epoch"] if history else 0


def stop_here(message="time budget reached"):
    print(f"\u23f8 {message} -- Runtime -> Disconnect and delete runtime, then Run all")
    return True


stopped = False
val_loader = itransformer.make_loader(X[is_val], y[is_val], BATCH_SIZE * 2)
main = {}
for seed in SEEDS:
    run, column = f"main-evening_seed{seed}", f"evening_seed{seed}"
    if registry.done(run):
        main[column] = origins.merge(load_frame(run), on=["city", "date"], how="left")[HORIZONS].to_numpy()
        print(f"\u2713 {run} already finished")
        continue
    if stopped or time.monotonic() > deadline:
        stopped = stop_here()
        break
    print(f"\n\u25b6 {run}")
    started = time.monotonic()
    model, history, outcome = fit(seed, X[is_train], y[is_train], val_loader, checkpoint=f"checkpoints/07/{run}.pt")
    if outcome == "out_of_time":
        stopped = stop_here("saved partway")
        break
    main[column] = itransformer.predict(model, X[scored], device)
    minutes = round((time.monotonic() - started) / 60, 2)
    save_run(run, {"seed": seed, "outcome": outcome, "best_epoch": best_epoch(history),
                   "epochs_run": len(history), "minutes": minutes, "history": history},
             frames={f"{run}.parquet": origins.assign(**{h: main[column][:, i] for i, h in enumerate(HORIZONS)})},
             weights=model.state_dict())
    print(f"  \u2191 recorded -- {outcome}, best epoch {best_epoch(history)}, {minutes} min")

# Gradient boosting reads the evening as columns. Its samples do not depend on the
# protocol, only their labels do, so they are built once and reused by every fold.
samples = windows.usable(windows.make_supervised(daily, features=WITH))
features = windows.feature_matrix(samples)
first_origin = pd.Timestamp(config.TRAIN_END_DATE) - pd.Timedelta(days=len(HORIZONS))
gbm_main = None
if registry.done("main-gbm-evening"):
    gbm_main = load_frame("main-gbm-evening")
    print("\u2713 main-gbm-evening already finished")
elif not stopped and time.monotonic() <= deadline:
    print("\n\u25b6 main-gbm-evening")
    started = time.monotonic()
    booster = baselines.fit_gradient_boosting(samples[samples.split == "train"], features)
    rows = samples[samples.date >= first_origin].copy()
    rows["gbm evening"] = baselines.predict_gradient_boosting(booster, rows, features).to_numpy()
    gbm_main = rows[KEYS + ["gbm evening"]]
    minutes = round((time.monotonic() - started) / 60, 2)
    save_run("main-gbm-evening", {"features": len(features), "minutes": minutes},
             frames={"main-gbm-evening.parquet": gbm_main})
    print(f"  \u2191 recorded -- {len(features)} columns, {minutes} min")
else:
    stopped = stop_here()

# Chronos-2 with the evening as past covariates, over every 2024-25 forecast at once.
fm_main = None
if registry.done("fm-chronos-2-evening"):
    fm_main = load_frame("fm-chronos-2-evening")
    print("\u2713 fm-chronos-2-evening already finished")
elif not stopped and time.monotonic() <= deadline:
    print(f"\n\u25b6 fm-chronos-2-evening  ({FM_MODEL}, the evening as past covariates)")
    started = time.monotonic()
    filled = sequences.fill_gaps(daily, variates, sequences.pooled_training_medians(daily, variates))
    enable_progress_bars()
    pipeline = foundation.load(FM_MODEL, device)
    disable_progress_bars()
    quantiles = foundation.forecast(pipeline, filled, origins, sequences.TARGET, EVENING, length=FM_CONTEXT,
                                    horizon=len(HORIZONS), levels=(0.1, 0.5, 0.9), batch_size=1024)
    del pipeline
    if device.type == "cuda":
        torch.cuda.empty_cache()
    fm_main = pd.concat([origins.assign(horizon=h + 1, **{"chronos-2-evening median": quantiles[:, h, 1],
                                                          "chronos-2-evening q90": quantiles[:, h, 2]})
                         for h in range(len(HORIZONS))], ignore_index=True)
    minutes = round((time.monotonic() - started) / 60, 2)
    save_run("fm-chronos-2-evening", {"model_id": FM_MODEL, "context": FM_CONTEXT, "minutes": minutes},
             frames={"fm-chronos-2-evening.parquet": fm_main})
    print(f"  \u2191 recorded -- {len(origins):,} forecasts x {len(HORIZONS)} horizons, {minutes} min")
else:
    stopped = stop_here()

## C · Leave-one-city-out, with the evening
Notebook 06's folds again: each city held out in turn, the windows rebuilt with it outside the training split, three seeds of the all-token head trained on the other fourteen cities' 2019–2023 and early-stopped on their 2024, and gradient boosting refitted on the same rows — all now reading the evening. The held-out city's 2024–2025 is what gets scored.

In [ ]:
unlabelled = samples.drop(columns="split")
for city in cities:
    run = fold_run(city)
    if registry.done(run):
        print(f"\u2713 {run} already finished")
        continue
    if stopped or time.monotonic() > deadline:
        stopped = stop_here()
        break
    print(f"\n\u25b6 {run}")
    started = time.monotonic()
    labels = assemble.leave_one_out_split(daily, city)
    Xf, yf, indexf = sequences.build_sequences(daily.assign(split=labels), features=WITH)
    train_rows, val_rows, test_rows = ((indexf.split == s).to_numpy() for s in ("train", "val", "test_loco"))
    keep = val_rows | test_rows
    fold_val = itransformer.make_loader(Xf[val_rows], yf[val_rows], BATCH_SIZE * 2)
    forecasts, runs, outcome, epochs = {}, [], None, []
    for seed in SEEDS:
        model, history, outcome = fit(seed, Xf[train_rows], yf[train_rows], fold_val, log=lambda *a: None)
        if outcome == "out_of_time":
            break
        forecasts[f"evening_seed{seed}"] = itransformer.predict(model, Xf[keep], device)
        runs.append({"seed": seed, "outcome": outcome, "best_epoch": best_epoch(history),
                     "epochs_run": len(history)})
        epochs.append(str(best_epoch(history)))
    if outcome == "out_of_time":
        stopped = stop_here("time budget reached mid-fold; this fold starts again next session")
        break
    frame = gap.long_frame(indexf[keep], yf[keep], Xf[keep, -1, target_index], forecasts)
    by_target = daily[["city", "date"]].assign(split=labels).rename(columns={"date": "target_date"})
    fold_samples = unlabelled.merge(by_target, on=["city", "target_date"], how="left")
    booster = baselines.fit_gradient_boosting(fold_samples[fold_samples.split == "train"], features)
    rows = fold_samples[fold_samples.split.isin(["val", "test_loco"])].copy()
    rows["gbm evening"] = baselines.predict_gradient_boosting(booster, rows, features).to_numpy()
    frame = frame.merge(rows[KEYS + ["gbm evening"]], on=KEYS, how="left")
    minutes = round((time.monotonic() - started) / 60, 2)
    save_run(run, {"city": city, "runs": runs, "windows": int(keep.sum()), "minutes": minutes},
             frames={f"{run}.parquet": frame})
    print(f"  best epochs {', '.join(epochs)};  \u2191 recorded with gradient boosting, {minutes} min")

## D · With and without, side by side
Notebook 06's saved forecasts — the families without the evening, and persistence — are joined to this notebook's, and every forecaster is scored on the rows all of them share. **Average precision** (how well a forecast ranks the severe days, no alarm level involved) is the measure; **onsets** are the severe target days that were not yet severe on the day the forecast was made. Alarm F1 uses notebook 06's rule (each fold's alarm level tuned on its validation rows) and is shown for continuity, with the onsets it caught.

In [ ]:
before_main = by_day(store.read_df("processed/gap_main_forecasts.parquet"))
before_loco = by_day(store.read_df("processed/gap_loco_forecasts.parquet"))
LINE = config.RARE_AQI_THRESHOLD
FAMILIES = {"all-token head": ([f"plain_seed{s}" for s in SEEDS], [f"evening_seed{s}" for s in SEEDS]),
            "gradient boosting": (["gbm"], ["gbm evening"]),
            "chronos-2": (["chronos-2 median"], ["chronos-2-evening median"])}


def with_fm(frame):
    if fm_main is None:
        return frame
    return frame.merge(fm_main[KEYS + ["chronos-2-evening median", "chronos-2-evening q90"]], on=KEYS, how="left")


after_main = gap.long_frame(index[scored], y[scored], np.zeros(int(scored.sum())), main)
after_main = after_main.drop(columns=["persistence", "y", "split"])
if gbm_main is not None:
    after_main = after_main.merge(gbm_main, on=KEYS, how="left")
combined_main = with_fm(before_main.merge(after_main, on=KEYS, how="inner"))
folds = {}
for city in cities:
    if registry.done(fold_run(city)):
        after = load_frame(fold_run(city)).drop(columns=["persistence", "y", "split"])
        before = before_loco[before_loco.fold == city].drop(columns="fold")
        folds[city] = with_fm(before.merge(after, on=KEYS, how="inner"))

rows_of = [("persistence", ["persistence"])] + [(f"{family}{suffix}", columns)
            for family, pair in FAMILIES.items() for suffix, columns in zip(("", " + evening"), pair)]
compared = [c for _, cols in rows_of for c in cols]
main_rows = combined_main.dropna(subset=[c for c in compared if c in combined_main])
loco_frames = {c: f.dropna(subset=[x for x in compared if x in f]) for c, f in folds.items()}
tested = (pd.concat([f[f.split == "test_loco"] for f in loco_frames.values()], ignore_index=True)
          if loco_frames else pd.DataFrame())
onsets = lambda frame: frame[frame.persistence <= LINE]


def onset_hits(column):
    """Onsets caught across the folds, each fold's alarm level tuned on its own validation rows."""
    hits = 0
    for f in loco_frames.values():
        level = alarms.tune_alarm(f.loc[f.split == "val", "y"], f.loc[f.split == "val", column])
        test = onsets(f[f.split == "test_loco"])
        hits += int(((test.y > LINE) & (test[column] > level)).sum())
    return hits


table = []
for name, columns in rows_of:
    if not all(c in main_rows for c in columns):
        continue
    unseen = main_rows[main_rows.split == "test_unseen"]
    row = {"forecaster": name,
           "AP, 5 unseen cities 2025": gap.average_precision(unseen, columns),
           "onset AP, 5 unseen": gap.average_precision(onsets(unseen), columns)}
    if len(tested) and all(c in tested for c in columns):
        row.update({"AP, held out": gap.average_precision(tested, columns),
                    "onset AP, held out": gap.average_precision(onsets(tested), columns),
                    "AP, held out without Delhi": gap.average_precision(tested[tested.city != "Delhi"], columns),
                    "RMSE, held out": float(np.mean([np.sqrt(np.mean((tested[c] - tested.y) ** 2)) for c in columns])),
                    "alarm F1, held out": float(np.mean([gap.score_folds(loco_frames, c)[0]["f1"] for c in columns])),
                    "onsets caught": float(np.mean([onset_hits(c) for c in columns]))})
    table.append(row)
table = pd.DataFrame(table).set_index("forecaster")
if len(tested):
    held_onsets = int((onsets(tested).y > LINE).sum())
    print(f"Held-out cities, 2024-25: {int((tested.y > LINE).sum())} severe targets, {held_onsets} of them onsets")
display(table.round(3))

## E · Does the evening help?
The measure fixed at the top: for each family, pooled leave-one-city-out average precision with the evening minus without, with a 95% interval and a p-value from resampling whole held-out cities, Holm-corrected across the three families. The same difference on onsets alone is shown beside it — it is where the evening was expected to matter — but it is secondary and is not what decides.

In [ ]:
if not len(tested):
    print("Leave-one-city-out has not finished yet -- the verdict needs it. Run all again to continue.")
else:
    rows, onset_sign = [], {}
    for family, (without, with_) in FAMILIES.items():
        if not all(c in tested for c in without + with_):
            continue
        d, low, high, p = gap.paired_ap_difference(tested, with_, without, draws=BOOTSTRAP_DRAWS)
        od, olow, ohigh, _ = gap.paired_ap_difference(onsets(tested), with_, without, draws=BOOTSTRAP_DRAWS)
        onset_sign[family] = "better" if olow > 0 else "worse" if ohigh < 0 else "no clear change"
        rows.append({"family": family, "AP without": gap.average_precision(tested, without),
                     "AP with": gap.average_precision(tested, with_), "difference": d,
                     "95% interval": f"[{low:+.3f}, {high:+.3f}]", "p": p,
                     "onset difference": od, "onset 95% interval": f"[{olow:+.3f}, {ohigh:+.3f}]"})
    verdict = pd.DataFrame(rows).set_index("family")
    verdict.insert(verdict.columns.get_loc("p") + 1, "p (Holm)", gap.holm(verdict["p"]))
    verdict["evening helps"] = np.where((verdict["p (Holm)"] < 0.05) & (verdict["difference"] > 0), "yes", "no")
    display(verdict.round(3))
    helped = verdict.index[verdict["evening helps"] == "yes"].tolist()
    if helped:
        sizes = ", ".join(f"{f} ({verdict.loc[f, 'difference']:+.3f})" for f in helped)
        print(f"\nThe evening helps, corrected p below 0.05: {sizes} in average precision. "
              "Read the size, not only the verdict: a small difference every city shares can clear the bar.")
    else:
        print("\nThe evening does not clear the bar for any family: whatever it carries, the daily "
              "features already carry, or it does not reach a city the model has never seen.")
    print("On onsets alone (interval for the difference): "
          + "; ".join(f"{f} {s}" for f, s in onset_sign.items()) + ".")

## F · Save
Both protocols' forecasts with and without the evening, and the score table, for the paper's tables.

In [ ]:
if SAVE_TO_HF:
    store.upload_df(combined_main, "processed/evening_main_forecasts.parquet",
                    "Notebook 07: main-split forecasts with and without the evening features")
    if folds:
        store.upload_df(pd.concat([f.assign(fold=c) for c, f in folds.items()], ignore_index=True),
                        "processed/evening_loco_forecasts.parquet",
                        "Notebook 07: leave-one-city-out forecasts with and without the evening features")
    store.upload_df(table.reset_index(), "processed/evening_scores.csv",
                    "Notebook 07: scores with and without the evening features")
    print("\u2191 processed/evening_main_forecasts.parquet, processed/evening_scores.csv"
          + (", processed/evening_loco_forecasts.parquet" if folds else ""))
print("\nNext: notebook 08 gathers notebooks 03-07 into the paper's tables.")